In [26]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [27]:
filename = "001.queryVentes.csv"
df = pd.read_csv(filename)

In [28]:
df.shape

(85674, 20)

In [19]:
df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,12:04:22.555,DONE,3.583788e+12,TONGS FEMME 100 NOIR,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,23.1,NaN,NaN
1,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,18:32:38.901,DONE,3.608439e+12,CASQUETTE ENFANT -MH100,1,NaN,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,23.1,NaN,NaN
2,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-11,12:11:15.59,DONE,3.583788e+12,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,NaN,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,24.6,NaN,NaN


In [20]:
ORDER_COL = "ORDER ID (TICKET DE CAISSE)"
PRODUCT_COL = "NOM DU PRODUIT"
df = (
    df[df["STATUT"].str.upper() == "DONE"][[ORDER_COL, PRODUCT_COL]]
    .dropna()
    .drop_duplicates(subset = [ORDER_COL, PRODUCT_COL])
)

df.shape

(70800, 2)

In [22]:
basket = (
    df.assign(value=1)
    .pivot_table(
        index=ORDER_COL,
        columns=PRODUCT_COL,
        values="value",
        aggfunc="max",
        fill_value=0
    )          
    .astype(bool)
)

In [23]:
basket

NOM DU PRODUIT,1 kit couverts bleu nuit (BINIKIT),100g Chips truffe MPG,100g choc noir MPG M. Havelaar,"5 minutes pour s'endormir, Histoire d'animaux","5 minutes pour s'endormir, Petits super-héros","5 minutes pour s'endormir, Super-Héros","5 minutes pour s'endormir, Un duo inséparable",5 puzzle Disney Princesses,6 barres chocolat 125g,6 barres de céréales au chocolat (125g),...,WRAP VEGGIE ŒUF PARMESAN TOMATES,YAOURT FRAISE FRAMBOISE,YAOURT FRAISE Framboise,YAOURT NATURE 125G,chaussures aquatiques enfant - turquoise,lipton Ice tea,lunette de natation enfant bleue,polaire grise Homme taille XL,serviette de plage bleue 145 x 85 cm,sprite 50cl
ORDER ID (TICKET DE CAISSE),,,,,,,,,,,,,,,,,,,,,
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56586,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
56587,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
56588,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [9]:
frequent_itemsets = apriori(
    basket,
    min_support=0.01,   # à ajuster selon ton volume
    use_colnames=True
)

In [10]:
frequent_itemsets.shape

(38, 2)

In [11]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [12]:
rules.shape

(0, 14)

In [13]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [14]:
rules[
    (rules["confidence"] >= 0.20) &
    (rules["lift"] >= 1.20)
].sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [25]:
df.columns

Index(['ORDER ID (TICKET DE CAISSE)', 'NOM DU PRODUIT'], dtype='object')

In [35]:
df[["NOM BOUTIQUE"]].groupby("NOM BOUTIQUE").agg({"NOM BOUTIQUE" : "count"})


,NOM BOUTIQUE
NOM BOUTIQUE,
Ibis budget Nice,5903
Ibis budget Strasbourg Centre République,4930
Mercure Paris Montmartre Sacré-Cœur,11766
Novotel Megève Mont-Blanc,4989
Novotel Paris Tour Eiffel,55962
Novotel Porte d’Italie,2124


In [36]:
df[["DATE"]].groupby("DATE").agg({"DATE" : "count"})

,DATE
DATE,
2023-08-10,2
2023-08-11,1
2023-08-12,2
2023-08-13,4
2023-08-15,1
...,...
2026-04-03,167
2026-04-04,192
2026-04-05,201
